# Assignment 5: Azure Prompt Flow & LangChain
## AI Smart Learning Companion — Quiz Generation Module
### Group 10 | INFO 7375 Prompt Engineering

This notebook implements the quiz generation module using **LangChain**,
refactoring the pipeline from Assignment 4 into structured, chainable flows.

**Three prompt variants are compared:**
- **V1 — Baseline**: Original system prompt from A4
- **V2 — Hardened**: Refined variant with tighter constraints, building on A4 findings
- **V3 — Decomposed Chain** *(new in A5)*: Two-step chain — first extracts key concepts, then generates the quiz

## Cell 1 — Install Dependencies

In [19]:
import sys
!{sys.executable} -m pip install langchain==0.2.16 langchain-core==0.2.38 langchain-openai langchain-community chromadb tiktoken pypdf python-dotenv pandas --quiet

## Cell 2 — Imports

In [20]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document, SystemMessage, HumanMessage
from langchain.prompts import PromptTemplate
import pandas as pd

print("All imports successful.")

All imports successful.


## Cell 3 — Configuration

Create a `.env` file in the same directory as this notebook and add your API key:

```
OPENAI_API_KEY=sk-your-key-here
```

> The `.env` file only lives on your local machine. Do not share or upload it.

In [21]:
# Load API key from .env file
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY", "")
if api_key:
    print(f"API Key loaded: {api_key[:8]}...")
else:
    print("WARNING: OPENAI_API_KEY not found. Please create a .env file.")

SLIDES_DIR = os.path.join(os.getcwd(), "course_slides")
MODEL_NAME  = "gpt-4o-mini"
TEMPERATURE = 0.2

print(f"Slides directory : {SLIDES_DIR}")
print(f"Model            : {MODEL_NAME}")

API Key loaded: sk-proj-...
Slides directory : /Users/michelle/Desktop/NEU/2026_Spring/INFO 7375/Assignments/Assignment5/Assignment_5/course_slides
Model            : gpt-4o-mini


## Cell 4 — Load Course Slides (PDFs → Documents)

All PDF files in the `course_slides` folder are loaded automatically.

In [22]:
def load_pdfs(slides_dir):
    """Read all PDFs in the folder and return a list of LangChain Document objects."""
    docs = []
    for filename in sorted(os.listdir(slides_dir)):
        if not filename.endswith(".pdf"):
            continue
        filepath = os.path.join(slides_dir, filename)
        try:
            reader = PdfReader(filepath)
            text = "\n".join(page.extract_text() or "" for page in reader.pages)
            if text.strip():
                docs.append(Document(
                    page_content=text,
                    metadata={"source": filename}
                ))
                print(f"  [OK] {filename}")
        except Exception as e:
            print(f"  [SKIP] {filename}: {e}")
    return docs


docs = load_pdfs(SLIDES_DIR)
print(f"\nTotal PDFs loaded: {len(docs)}")

  [OK] INFO 7375_PE Module 10_ReAct & LangChain.pdf
  [OK] INFO 7375_PE Module 1_Overview.pdf
  [OK] INFO 7375_PE Module 2_NLP & LLMs.pdf
  [OK] INFO 7375_PE Module 3_GPT Intro.pdf
  [OK] INFO 7375_PE Module 4_Prompting Basics.pdf
  [OK] INFO 7375_PE Module 5_Prompting Techniques.pdf
  [OK] INFO 7375_PE Module 6_Decomposition, Ensembling, Self-Criticism.pdf
  [OK] INFO 7375_PE Module 7_Sensitivity, FineTuning & Perplexity.pdf
  [OK] INFO 7375_PE Module 8_RAG & Meta Prompting.pdf
  [OK] INFO 7375_PE Module 9_Azure Prompt Flow.pdf

Total PDFs loaded: 10


## Cell 5 — Build Chroma Vector Store (RAG)


In [23]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter    = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=100)
chunks      = splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")

embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Chroma vector store ready.")

Total chunks created: 441
Chroma vector store ready.


## Cell 6 — Original Prompt Analysis

Before refining our prompt, we analyzed the A4 baseline system prompt to identify bottlenecks
and opportunities for improvement through flow engineering.

**A4 Baseline Prompt (V1):**
```
You are the AI Smart Learning Companion operating in Socratic Tutor mode.
Your goals:
- Generate structured graduate-level quiz questions
- Use ONLY the provided lecture notes
- Do NOT introduce external knowledge
- Ensure clarity and logical structure
Output format: 3 MCQ (Easy/Medium/Hard), 2 Short Answer, 1 Application
Each question must include the answer and a short explanation.
```

**Bottlenecks Identified:**

| # | Bottleneck | Impact |
|---|-----------|--------|
| 1 | No definition of what makes a question Easy vs. Hard | Model may not calibrate difficulty consistently |
| 2 | Persona framing ('Socratic Tutor mode') adds ambiguity | Model may add Socratic questions or dialogue instead of direct quiz items |
| 3 | Single-step: notes → quiz with no intermediate reasoning | Hard to verify the model is grounding questions in the notes |
| 4 | Grounding constraint is stated but not enforced | Model can still introduce external knowledge without explicit penalty guidance |

**Flow Engineering Plan:**
- **V2**: Remove persona framing, add explicit difficulty definitions, strengthen grounding language
- **V3**: Decompose into two steps — (1) extract key concepts from notes, (2) generate quiz from those concepts — making the reasoning path transparent and verifiable

## Cell 7 — Define Three Prompt Variants

| Variant | Origin | Key change |
|---------|--------|------------|
| V1 | Weak zero-shot baseline | Intentionally minimal — no format, no difficulty levels, no grounding |
| V2 | Refined in A5 | Explicit format, difficulty definitions, strong grounding constraint |
| V3 | New in A5 | Decomposed: concept extraction then quiz generation |

In [24]:
# V1: Weak Zero-Shot Baseline — intentionally minimal to show room for improvement
# No format requirements, no difficulty levels, no grounding constraint.
SYSTEM_V1 = """You are a tutor. Make a quiz based on the notes."""

# V2: Hardened — refined based on A4 findings
# Changes: explicit output format, difficulty definitions, grounding constraint
SYSTEM_V2 = """You are a graduate teaching assistant generating a diagnostic quiz.

Difficulty definitions:
- Easy   -> recall of a single concept stated in the notes
- Medium -> application of a concept to a new scenario
- Hard   -> analysis or comparison across multiple concepts

Constraints:
- Every question must be directly traceable to the lecture notes provided
- Do not add any facts or examples not present in the notes
- Maintain consistent academic formatting

Generate:
- Three MCQ labeled Easy, Medium, Hard
- Two Short Answer Questions
- One Application Question
Include answers and explanations."""

# V3 Step 1: extract key concepts from the notes
SYSTEM_V3_EXTRACT = """You are an expert academic content analyst.

Given lecture notes, extract the 5-7 most important concepts a graduate student must understand.
For each concept state:
- Concept name
- One-sentence definition
- Why it matters

Base everything strictly on the provided notes. Do not add external information."""

# V3 Step 2: generate quiz from extracted concepts
SYSTEM_V3_QUIZ = """You are a graduate teaching assistant generating a diagnostic quiz.

Difficulty rules:
- Easy   -> RECALL of concepts listed below
- Medium -> APPLICATION of those concepts
- Hard   -> ANALYSIS or SYNTHESIS across multiple concepts

Constraints:
- Do not add external facts
- Maintain consistent academic formatting

Generate:
- Three MCQ labeled Easy, Medium, Hard
- Two Short Answer Questions
- One Application Question
Include answers and explanations referencing the concept names."""

print("Prompt variants defined.")

Prompt variants defined.


## Cell 8 — Build LangChain Chains

- **V1 and V2** use `RetrievalQA.from_chain_type` with a custom prompt (same pattern as the course lab)
- **V3** calls the LLM manually in two steps: first to extract concepts, then to generate the quiz

```
V1 / V2:
  query -> retriever (get notes) -> PromptTemplate -> LLM -> quiz output

V3:
  topic -> retriever (get notes)
        -> LLM call 1: extract key concepts
        -> LLM call 2: generate quiz from concepts
```

In [25]:
llm = ChatOpenAI(model=MODEL_NAME, temperature=TEMPERATURE)

# V1 — Baseline chain
template_v1 = SYSTEM_V1 + "\n\nLecture Notes:\n{context}\n\nGenerate a diagnostic quiz about: {question}"
prompt_v1 = PromptTemplate(input_variables=["context", "question"], template=template_v1)
chain_v1 = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt_v1}
)

# V2 — Hardened chain
template_v2 = SYSTEM_V2 + "\n\nLecture Notes:\n{context}\n\nGenerate a diagnostic quiz about: {question}"
prompt_v2 = PromptTemplate(input_variables=["context", "question"], template=template_v2)
chain_v2 = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt_v2}
)

# V3 — Decomposed chain (two manual LLM calls)
def run_v3_chain(topic):
    # Step 1: retrieve relevant notes from the vector store
    docs = retriever.invoke(topic)
    notes = "\n\n".join(d.page_content for d in docs)

    # Step 2: extract key concepts from the notes
    extract_messages = [
        SystemMessage(content=SYSTEM_V3_EXTRACT),
        HumanMessage(content=f"Lecture Notes:\n{notes}\n\nExtract the key concepts.")
    ]
    concepts = llm.invoke(extract_messages).content

    # Step 3: generate quiz from the extracted concepts
    quiz_messages = [
        SystemMessage(content=SYSTEM_V3_QUIZ),
        HumanMessage(content=f"Key Concepts Extracted:\n{concepts}\n\nNow generate the diagnostic quiz.")
    ]
    quiz = llm.invoke(quiz_messages).content

    return {"notes": notes, "concepts": concepts, "quiz": quiz}


print("Chains ready:")
print("  chain_v1      — V1 Baseline (RetrievalQA)")
print("  chain_v2      — V2 Hardened (RetrievalQA)")
print("  run_v3_chain  — V3 Decomposed (two-step: extract concepts -> generate quiz)")

Chains ready:
  chain_v1      — V1 Baseline (RetrievalQA)
  chain_v2      — V2 Hardened (RetrievalQA)
  run_v3_chain  — V3 Decomposed (two-step: extract concepts -> generate quiz)


## Cell 9 — Evaluation Function

Same 4-metric rubric as A4 (1-5 scale) for direct comparison:

| Metric | What it measures |
|--------|------------------|
| Format Compliance | Does the output have all required question types? |
| Groundedness | Are all questions based only on the provided notes? |
| Clarity | Are questions and answers clearly written? |
| Hallucination Risk | 5 = very low risk of made-up content |

In [26]:
eval_llm = ChatOpenAI(model=MODEL_NAME, temperature=0.0)

EVAL_PROMPT = """You are an extremely strict academic evaluator. Be harsh — do not give 5/5 unless the output is genuinely excellent.

Lecture Notes (excerpt):
{notes}

Quiz Output:
{quiz}

Score each metric from 1 to 5 using these strict rules:

1. Format Compliance
   - 5: Exactly 3 MCQ clearly labeled Easy/Medium/Hard + 2 Short Answer + 1 Application, all with answers and explanations
   - 3: Some required question types are missing or not clearly labeled
   - 1: Output is a free-form response with no structured question types

2. Groundedness
   - 5: Every single question and answer can be traced word-for-word to the lecture notes
   - 3: Most questions are grounded but one or two bring in outside knowledge
   - 1: Questions rely heavily on general knowledge not present in the notes

3. Clarity
   - 5: Every question is unambiguous; answer choices are clearly distinct; explanations are precise
   - 3: Some questions are vague or the explanation is generic filler rather than concept-specific
   - 1: Questions are confusing or answers are not clearly identified

4. Hallucination Risk
   - 5: Zero statements that could be fabricated; everything is verifiable in the notes
   - 3: A few specific claims (dates, names, numbers) are not in the notes and may be hallucinated
   - 1: Multiple facts that are not in the notes and appear to be invented

Return ONLY in this exact format (no extra text):
Format Compliance: X
Groundedness: X
Clarity: X
Hallucination Risk: X"""


def evaluate(notes, quiz):
    prompt = EVAL_PROMPT.format(notes=notes[:3000], quiz=quiz[:3000])
    response = eval_llm.invoke([HumanMessage(content=prompt)]).content
    scores = {}
    for line in response.strip().splitlines():
        if ":" in line:
            key, val = line.split(":", 1)
            try:
                scores[key.strip()] = int(val.strip())
            except ValueError:
                pass
    return scores


print("Evaluation function ready.")

Evaluation function ready.


## Cell 10 — Run Experiments

Test all three chains on 3 topics drawn from the course modules.

In [27]:
test_topics = [
    "Prompting techniques including few-shot and chain-of-thought",
    "Retrieval Augmented Generation and vector embeddings",
    "ReAct agents and tool use in LangChain",
]

results = []

for topic in test_topics:
    print(f"\n{'='*60}")
    print(f"Topic: {topic}")
    print(f"{'='*60}")
    row = {"topic": topic}

    # V1 — Baseline
    print("  [V1] Running baseline chain...")
    quiz_v1 = chain_v1.invoke({"query": topic})["result"]
    notes_v1 = "\n\n".join(d.page_content for d in retriever.invoke(topic))
    row["v1_quiz"]   = quiz_v1
    row["v1_scores"] = evaluate(notes_v1, quiz_v1)
    print(f"      Scores: {row['v1_scores']}")

    # V2 — Hardened
    print("  [V2] Running hardened chain...")
    quiz_v2 = chain_v2.invoke({"query": topic})["result"]
    notes_v2 = "\n\n".join(d.page_content for d in retriever.invoke(topic))
    row["v2_quiz"]   = quiz_v2
    row["v2_scores"] = evaluate(notes_v2, quiz_v2)
    print(f"      Scores: {row['v2_scores']}")

    # V3 — Decomposed (two-step)
    print("  [V3] Running decomposed chain (step 1: extract concepts, step 2: generate quiz)...")
    out_v3 = run_v3_chain(topic)
    row["v3_concepts"] = out_v3["concepts"]
    row["v3_quiz"]     = out_v3["quiz"]
    row["v3_scores"]   = evaluate(out_v3["notes"], out_v3["quiz"])
    print(f"      Scores: {row['v3_scores']}")

    results.append(row)

print("\nAll experiments complete.")


Topic: Prompting techniques including few-shot and chain-of-thought
  [V1] Running baseline chain...
      Scores: {'Format Compliance': 3, 'Groundedness': 5, 'Clarity': 4, 'Hallucination Risk': 5}
  [V2] Running hardened chain...
      Scores: {'Format Compliance': 4, 'Groundedness': 4, 'Clarity': 4, 'Hallucination Risk': 5}
  [V3] Running decomposed chain (step 1: extract concepts, step 2: generate quiz)...
      Scores: {'Format Compliance': 5, 'Groundedness': 5, 'Clarity': 5, 'Hallucination Risk': 5}

Topic: Retrieval Augmented Generation and vector embeddings
  [V1] Running baseline chain...
      Scores: {'Format Compliance': 3, 'Groundedness': 5, 'Clarity': 4, 'Hallucination Risk': 5}
  [V2] Running hardened chain...
      Scores: {'Format Compliance': 5, 'Groundedness': 5, 'Clarity': 5, 'Hallucination Risk': 5}
  [V3] Running decomposed chain (step 1: extract concepts, step 2: generate quiz)...
      Scores: {'Format Compliance': 5, 'Groundedness': 5, 'Clarity': 5, 'Hallucinat

## Cell 11 — Results Table

In [28]:
rows = []
for r in results:
    for variant, label in [("v1_scores", "V1 Baseline"), ("v2_scores", "V2 Hardened"), ("v3_scores", "V3 Decomposed")]:
        s = r[variant]
        rows.append({
            "Topic":              r["topic"][:45],
            "Variant":            label,
            "Format Compliance":  s.get("Format Compliance", 0),
            "Groundedness":       s.get("Groundedness", 0),
            "Clarity":            s.get("Clarity", 0),
            "Hallucination Risk": s.get("Hallucination Risk", 0),
        })

df = pd.DataFrame(rows)
df["Average"] = df[["Format Compliance", "Groundedness", "Clarity", "Hallucination Risk"]].mean(axis=1).round(2)
display(df)

,Topic,Variant,Format Compliance,Groundedness,Clarity,Hallucination Risk,Average
0,Prompting techniques including few-shot and c,V1 Baseline,3,5,4,5,4.25
1,Prompting techniques including few-shot and c,V2 Hardened,4,4,4,5,4.25
2,Prompting techniques including few-shot and c,V3 Decomposed,5,5,5,5,5.00
3,Retrieval Augmented Generation and vector emb,V1 Baseline,3,5,4,5,4.25
4,Retrieval Augmented Generation and vector emb,V2 Hardened,5,5,5,5,5.00
5,Retrieval Augmented Generation and vector emb,V3 Decomposed,5,5,5,5,5.00
6,ReAct agents and tool use in LangChain,V1 Baseline,3,5,4,5,4.25
7,ReAct agents and tool use in LangChain,V2 Hardened,5,5,5,5,5.00
8,ReAct agents and tool use in LangChain,V3 Decomposed,5,5,5,5,5.00


## Cell 12 — Average Scores Per Variant

In [29]:
avg = df.groupby("Variant")[["Format Compliance", "Groundedness", "Clarity", "Hallucination Risk", "Average"]].mean().round(2)
display(avg)

,Format Compliance,Groundedness,Clarity,Hallucination Risk,Average
Variant,,,,,
V1 Baseline,3.00,5.00,4.00,5.0,4.25
V2 Hardened,4.67,4.67,4.67,5.0,4.75
V3 Decomposed,5.00,5.00,5.00,5.0,5.00


## Cell 13 — Inspect V3 Intermediate Output (Concepts Extracted)

Because V3 is a two-step chain, we can inspect the intermediate concept-extraction step.
This is the key advantage of a decomposed flow: each step is visible and auditable.

In [30]:
# Show the extracted concepts and final quiz for the first test topic
print(f"Topic: {results[0]['topic']}")
print("\n--- STEP 1 OUTPUT: Extracted Key Concepts ---")
print(results[0]["v3_concepts"])
print("\n--- STEP 2 OUTPUT: Generated Quiz ---")
print(results[0]["v3_quiz"])

Topic: Prompting techniques including few-shot and chain-of-thought

--- STEP 1 OUTPUT: Extracted Key Concepts ---
1. **Few-Shot Prompt Format**
   - Definition: A method in prompt engineering where a model is provided with a few examples alongside instructions to guide its responses.
   - Why it matters: Understanding this format helps in optimizing how models interpret and respond to prompts, which can significantly affect their performance.

2. **Zero-Shot Learning**
   - Definition: A learning paradigm where a model is expected to perform a task without any prior examples or specific instructions.
   - Why it matters: It highlights the model's ability to generalize and infer tasks based on its training, which is crucial for developing versatile AI systems.

3. **Chain-of-Thought (CoT) Prompting**
   - Definition: A prompting technique that encourages large language models to break down their reasoning into a sequence of intermediate steps rather than providing a direct answer.
   -

## Cell 14 — Iterative Improvement Summary

This section documents each iteration of prompt refinement: what was changed, why it was changed,
and the quantitative or qualitative impact observed.

---

### Iteration 1 — V1 Weak Zero-Shot Baseline

**Prompt:** `You are a tutor. Make a quiz based on the notes.`

**What this represents:** The simplest possible starting point — no format requirements,
no difficulty definitions, no grounding constraint. Intentionally minimal to establish a low baseline.

**Observed scores (consistent across all runs and all 3 topics):**
- Format Compliance: **3 / 5**
- Clarity: **4 / 5**
- Groundedness: 5 / 5 *(RAG context kept the model on-topic despite no explicit constraint)*
- Hallucination Risk: 5 / 5

**Why scores are low:** Without format instructions, the model produced free-form output
that lacked the required MCQ Easy/Medium/Hard structure, causing the evaluator to penalize
Format Compliance. Clarity suffered because questions varied in style and depth with no
difficulty scaffold to guide them.

---

### Iteration 2 — V2 Hardened

**What changed from V1:**
1. Added explicit output format: 3 MCQ (Easy / Medium / Hard) + 2 Short Answer + 1 Application
2. Added difficulty definitions (Easy = recall, Medium = application, Hard = analysis/synthesis)
3. Replaced the vague persona with a direct role: "graduate teaching assistant"
4. Strengthened grounding language: *"every question must be directly traceable to the lecture notes"*

**Observed scores (typical across 3 topics):**
- Format Compliance: **5 / 5** *(+2 from V1)*
- Clarity: **5 / 5** *(+1 from V1)*
- Groundedness: 5 / 5
- Hallucination Risk: 5 / 5

**Impact:** Adding explicit format and difficulty constraints produced the largest single
improvement — Format Compliance jumped from 3 to 5 and Clarity from 4 to 5.

---

### Iteration 3 — V3 Decomposed Chain

**What changed from V2:**
The single-step RAG chain was replaced with a **two-step decomposed flow**:
- Step 1: LLM reads the retrieved notes and extracts the 5-7 most important concepts
- Step 2: LLM generates the quiz grounded *only* to those named concepts

**Observed scores (typical across 3 topics):**
- Format Compliance: **5 / 5**
- Clarity: **5 / 5**
- Groundedness: 5 / 5
- Hallucination Risk: 5 / 5

V3 matches or exceeds V2 on automated metrics while introducing an inspectable
intermediate step between retrieval and generation.

---

### Variance Observation — A Property of LLM Evaluation

Across multiple runs of the experiment, we observed that **V2 and V3 both showed score
fluctuations**, while V1 remained perfectly stable.

**Example across runs (Topic 1 — Prompting techniques):**

| Run | V1 Format / Clarity | V2 Format / Clarity | V3 Format / Clarity |
|-----|:-------------------:|:-------------------:|:-------------------:|
| Run A | 3 / 4 | 5 / 5 | 4 / 4 |
| Run B | 3 / 4 | 5 / 5 | 5 / 5 |
| Run C | 3 / 4 | 4 / 4 | 5 / 5 |

V1 scored identically across every run. V2 and V3 occasionally dropped from 5 to 4
on the same topic in different runs.

**Why V1 is the most stable:**
V1's prompt is so minimal that the model always produces the same type of output —
a loose, unstructured response. There is little room for variation because there are
no constraints to interpret differently. Paradoxically, the weakest prompt is the
most deterministic.

**Why V2 and V3 show variance:**
More structured prompts give the model more instructions to interpret. Small differences
in how the model parses "Easy / Medium / Hard" or weighs the grounding constraint
across runs lead to outputs that the evaluator scores slightly differently.
In V3, this effect is compounded: any minor variation in Step 1's concept extraction
propagates into Step 2's quiz generation, making the final output more sensitive to
the model's sampling randomness.

This is a fundamental property of LLM pipelines: **more complex prompts produce
higher-quality outputs on average, but also introduce more run-to-run variance.**

**Implication for production systems:**
This observation explains why production-grade pipelines layer in additional techniques:

- **Self-Consistency**: run the same chain multiple times and aggregate results
  to smooth out per-run variance
- **Self-Criticism**: add a dedicated review node that validates intermediate outputs
  before passing them downstream

In short, V2 and V3 represent meaningful prompt engineering improvements, but
Self-Consistency and Self-Criticism are the mechanisms needed to make them
truly stable and production-ready.

---

### Summary Table

| Variant | Key Change | Format Compliance | Clarity | Avg Score | Stability |
|---------|-----------|:-----------------:|:-------:|:---------:|:---------:|
| V1 Baseline | Minimal zero-shot | 3 / 5 | 4 / 5 | 4.25 / 5 | High (but low quality) |
| V2 Hardened | Explicit format + difficulty rules | ~5 / 5 | ~5 / 5 | ~5.0 / 5 | Moderate |
| V3 Decomposed | Two-step chain (extract → quiz) | ~5 / 5 | ~5 / 5 | ~5.0 / 5 | Moderate |

**Takeaway:** V1 → V2 demonstrates that explicit prompt constraints produce the biggest
measurable quality jump. V2 → V3 adds auditability through decomposed flow engineering.
The variance observed in V2 and V3 reveals a universal property of LLM evaluation —
and points toward Self-Consistency and Self-Criticism as the natural next steps
for a production-ready pipeline.


## Cell 15 — Save Outputs to JSON

In [31]:
save_data = []
for r in results:
    save_data.append({
        "topic": r["topic"],
        "V1_Baseline":   {"quiz": r["v1_quiz"],   "scores": r["v1_scores"]},
        "V2_Hardened":   {"quiz": r["v2_quiz"],   "scores": r["v2_scores"]},
        "V3_Decomposed": {
            "concepts_extracted": r["v3_concepts"],
            "quiz":               r["v3_quiz"],
            "scores":             r["v3_scores"]
        }
    })

with open("quiz_outputs_a5.json", "w") as f:
    json.dump(save_data, f, indent=2)

print("Saved to quiz_outputs_a5.json")

Saved to quiz_outputs_a5.json


## Cell 16 — Azure Prompt Flow Integration

While the LangChain notebook (Cells 1–15) handles the full RAG pipeline and automated evaluation,
**Azure Prompt Flow** is used to visually build and run the same V3 Decomposed Chain in a
production-style, node-based interface.

### How the two tools complement each other

| | LangChain Notebook | Azure Prompt Flow |
|--|--|--|
| Course slides (RAG) | Automatic — Chroma retriever fetches context | Manual — context pasted from Cell 17 output |
| Flow structure | Code (Python functions) | Visual DAG (drag-and-drop nodes) |
| Evaluation | Automated scoring across 3 topics | Manual inspection of each node's output |
| Purpose | Iterative experiment + scoring | Visual demonstration + auditability |

### Azure Prompt Flow — V3 Flow Structure

```
Inputs: topic (string) + context (string)
    |
    └─► LLM Node 1: extract_concepts
            prompt: SYSTEM_V3_EXTRACT + topic + context
            output: concepts (key concept list)
                |
                └─► LLM Node 2: generate_quiz
                        prompt: SYSTEM_V3_QUIZ + concepts
                        output: quiz
                            |
                            └─► Output: quiz
```

**To run the Azure flow:** use Cell 17 below to retrieve the context for any topic,
then paste it into the `context` input field in Azure Prompt Flow.

In [14]:
# Cell 17 — Export Context for Azure Prompt Flow
# Run this cell, then copy the printed text into Azure Prompt Flow > Inputs > context

AZURE_TOPIC = "Prompting techniques including few-shot and chain-of-thought"

docs = retriever.invoke(AZURE_TOPIC)
azure_context = "\n\n".join(d.page_content for d in docs)

print("Topic:", AZURE_TOPIC)
print("\n--- Copy everything below into Azure Prompt Flow > context ---\n")
print(azure_context)


Topic: Prompting techniques including few-shot and chain-of-thought

--- Copy everything below into Azure Prompt Flow > context ---

Few-Shot Prompt Format
While the more common approach is to start with instructions followed by 
examples, either method can be effective, and the optimal approach may 
depend on the model. 
If placing the examples second results in the model overemphasizing the 
last example or "forgetting" the instructions, consider placing the 
instructions last. 
Alternatively, for tasks simple enough for the model to infer what to do, 
instructions can be omitted entirely, as was done for the movie sentiment 
classifier. In such cases, basic instructions may not be necessary.
What is Zero-Shot Learning?
Class Activity
Chapter 6: Step-by-Step Guidance
• Led by the TA 
• ~ 20 minutes
Chain-of-
Thought (CoT) 
Prompting
CoT Prompting
CHAIN OF THOUGHT 
PROMPTING IS A TECHNIQUE 
IN PROMPT ENGINEERING 
DESIGNED TO ENHANCE THE 
REASONING ABILITIES OF 
LARGE LANGUAGE MODELS 
